# Ejemplo: Usando la Conexión Centralizada

Este notebook demuestra cómo usar el módulo `conexion_sqlserver` en tus propios notebooks.

## Paso 1: Importar el módulo de conexión

In [1]:
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, SERVIDOR, BD_ORIGEN, BD_DESTINO, USUARIO
from conexion_sqlserver import ANIO_INICIO, ANIO_FIN

# Obtener los engines
engine_origen, engine_destino = obtener_engines()

print(f'\n=== CONFIGURACION ==="')
print(f'Servidor: {SERVIDOR}')
print(f'Usuario: {USUARIO}')
print(f'Rango temporal: {ANIO_INICIO} - {ANIO_FIN}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')

Conexion OK (origen): WideWorldImporters
Conexion OK (destino): WideWorldImportersDW2

=== CONFIGURACION ==="
Servidor: localhost\SQLEXPRESS02
Usuario: ingesta_reader
Rango temporal: 2010 - 2035
Origen: WideWorldImporters
Destino: WideWorldImportersDW2


## Paso 2: Usar los engines para lectura

In [17]:
import pandas as pd
from sqlalchemy import text

# Leer datos del servidor de origen
consulta = """
SELECT 
    CustomerID, CustomerName, CityID, DeliveryPostalCode
FROM Sales.Customers
"""

consulta = """
SELECT
    CustomerID,
    CustomerName,
    DeliveryCityID AS CityID,
    DeliveryPostalCode
FROM Sales.Customers
"""

df = pd.read_sql_query(consulta, engine_origen)
print(f'Se leyeron {len(df)} clientes:')
display(df)

Se leyeron 663 clientes:


,CustomerID,CustomerName,CityID,DeliveryPostalCode
0,1,Tailspin Toys (Head Office),19586,90410
1,2,"Tailspin Toys (Sylvanite, MT)",33475,90216
2,3,"Tailspin Toys (Peeples Valley, AZ)",26483,90205
3,4,"Tailspin Toys (Medicine Lodge, KS)",21692,90152
4,5,"Tailspin Toys (Gasport, NY)",12748,90261
...,...,...,...,...
658,1057,Ganesh Majumdar,25608,90760
659,1058,Jaroslav Fisar,31564,90708
660,1059,Jibek Juniskyzy,10483,90073
661,1060,Anand Mudaliyar,22090,90669


## Paso 3: Crear un esquema en el destino

In [9]:
# Crear esquema 'dw' si no existe
sql_crear_esquema = """
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'dw')
    EXEC('CREATE SCHEMA dw');
"""

with engine_destino.begin() as conn:
    conn.execute(text(sql_crear_esquema))
    print('Esquema dw creado o ya existente')

Esquema dw creado o ya existente


## Paso 4: Insertar datos en el destino

In [18]:
# Crear una tabla de ejemplo en el destino
sql_crear_tabla = """
IF OBJECT_ID('dw.clientes_ejemplo', 'U') IS NULL
BEGIN
    CREATE TABLE dw.clientes_ejemplo (
        customer_id INT PRIMARY KEY,
        customer_name NVARCHAR(100),
        city_id INT,
        postal_code NVARCHAR(20),
        fecha_carga DATETIME DEFAULT GETDATE()
    );
END;
"""

with engine_destino.begin() as conn:
    conn.execute(text(sql_crear_tabla))
    print('Tabla dw.clientes_ejemplo creada o ya existente')

Tabla dw.clientes_ejemplo creada o ya existente


## Paso 5: Insertar los datos leídos

In [19]:
# Renombrar columnas para que coincidan con la tabla destino
df_cargable = df.rename(columns={
    'CustomerID': 'customer_id',
    'CustomerName': 'customer_name',
    'CityID': 'city_id',
    'DeliveryPostalCode': 'postal_code'
})

# Insertar datos (reemplazar si existen)
try:
    df_cargable.to_sql(
        'clientes_ejemplo',
        engine_destino,
        schema='dw',
        if_exists='append',  # o 'replace' para sobrescribir
        index=False
    )
    print(f'Se insertaron {len(df_cargable)} registros en dw.clientes_ejemplo')
except Exception as e:
    print(f'Error al insertar: {e}')

Se insertaron 663 registros en dw.clientes_ejemplo


In [14]:
df_cargable.head()

,customer_id,customer_name,BillToCustomerID,CustomerCategoryID,BuyingGroupID,PrimaryContactPersonID,AlternateContactPersonID,DeliveryMethodID,city_id,PostalCityID,...,DeliveryRun,RunPosition,WebsiteURL,DeliveryAddressLine1,DeliveryAddressLine2,postal_code,DeliveryLocation_WKT,LastEditedBy,ValidFrom,ValidTo
0,1,Tailspin Toys (Head Office),1,3,1.0,1001,1002.0,3,19586,19586,...,,,http://www.tailspintoys.com,Shop 38,1877 Mittal Road,90410,POINT (-102.6201979 41.4972022),1,2013-01-01,9999-12-31 23:59:59.999999
1,2,"Tailspin Toys (Sylvanite, MT)",1,3,1.0,1003,1004.0,3,33475,33475,...,,,http://www.tailspintoys.com/Sylvanite,Shop 245,705 Dita Lane,90216,POINT (-115.8743507 48.7163356),1,2013-01-01,9999-12-31 23:59:59.999999
2,3,"Tailspin Toys (Peeples Valley, AZ)",1,3,1.0,1005,1006.0,3,26483,26483,...,,,http://www.tailspintoys.com/PeeplesValley,Unit 217,1970 Khandke Road,90205,POINT (-112.7271223 34.2689145),1,2013-01-01,9999-12-31 23:59:59.999999
3,4,"Tailspin Toys (Medicine Lodge, KS)",1,3,1.0,1007,1008.0,3,21692,21692,...,,,http://www.tailspintoys.com/MedicineLodge,Suite 164,967 Riutta Boulevard,90152,POINT (-98.580361 37.2811339),1,2013-01-01,9999-12-31 23:59:59.999999
4,5,"Tailspin Toys (Gasport, NY)",1,3,1.0,1009,1010.0,3,12748,12748,...,,,http://www.tailspintoys.com/Gasport,Unit 176,1674 Skujins Boulevard,90261,POINT (-78.5761394 43.1992244),1,2013-01-01,9999-12-31 23:59:59.999999


## Paso 6: Verificar los datos cargados

In [ ]:
# Leer desde la tabla destino para verificar
consulta_verificacion = """
SELECT customer_id, customer_name, postal_code, fecha_carga
FROM dw.clientes_ejemplo
ORDER BY customer_id
"""

df_verificado = pd.read_sql_query(consulta_verificacion, engine_destino)
print(f'Registros en dw.clientes_ejemplo: {len(df_verificado)}')
display(df_verificado)

## Paso 7: Ejecutar sentencias SQL complejas

In [ ]:
# Ejemplo: generar estadísticas
sql_estadisticas = """
SELECT 
    COUNT(*) AS total_registros,
    COUNT(DISTINCT customer_id) AS clientes_unicos,
    COUNT(DISTINCT city_id) AS ciudades
FROM dw.clientes_ejemplo
"""

with engine_destino.connect() as conn:
    resultado = conn.execute(text(sql_estadisticas)).mappings().first()
    print('\n=== ESTADISTICAS ===')
    for clave, valor in resultado.items():
        print(f'{clave}: {valor}')

## Notas Importantes

1. **Los engines son reutilizables**: Una vez obtenidos, puedes usarlos múltiples veces en el notebook
2. **Contextos transaccionales**: Usa `with engine.begin()` para transacciones o `with engine.connect()` para lectura
3. **Manejo de errores**: Siempre rodea operaciones de inserción con try/except
4. **Desempeño**: Para grandes volúmenes, considera usar `if_exists='append'` y batch inserts
5. **Auditoría**: La tabla destino incluye `fecha_carga` automáticamente para rastrear cuándo se insertó cada registro